In [50]:
"""
Sequential distillation factory search (Section III).

Implements the closed-form sequential phase condition (Theorem 2):

    Delta_Phi(i, j) = [
        2^(i+1) theta sigma_j sum_{w in W_j} C(n-j-i, w-i)
      + sum_{q=1}^{j-1} 2^(i+1) theta sigma_q sum_{w in W_q} C(n-q-i-1, w-i-1)
      - sum_{q=1}^{j-1} 2^i theta sigma_q sum_{w in W_q} C(n-q-i, w-i)
    ] mod 2*pi

with theta = pi/2^l. The disentangling condition requires Delta_Phi(i, j)
to be the same for all i (i.e. independent of i).

For each (l, n) we enumerate sequences (W_1, ..., W_J) with J = n - k_out
steps, where W_j is drawn from the arithmetic-progression family
{w == 1 mod s_j} for s_j in {1, 2, 3, 4}, independently per step.

Two passes: signs all +1, then signs in {+1, -1}^J.
"""

from math import comb, pi
from itertools import product


# --- Core verifier ------------------------------------------------------------

def delta_phi_over_theta(i, j, n, W_list, sigma_list):
    """
    Compute Delta_Phi(i, j) / theta as an INTEGER (the result of dividing
    Delta_Phi mod 2*pi by theta = pi/2^l, so this is an integer mod 2^(l+1)).

    The formula above multiplied by 2^l/pi (i.e. divided by theta).

    j is 1-indexed (matches the paper). W_list[q-1] is W_q for q in 1..j.
    """
    # Step-j contribution
    val = 0
    sigma_j = sigma_list[j - 1]
    Wj = W_list[j - 1]
    coeff_j = (1 << (i + 1)) * sigma_j  # 2^(i+1) * sigma_j
    val += coeff_j * sum(comb(n - j - i, w - i) for w in Wj
                          if 0 <= w - i <= n - j - i)

    # Earlier-step residuals
    for q in range(1, j):
        sigma_q = sigma_list[q - 1]
        Wq = W_list[q - 1]
        c_plus = (1 << (i + 1)) * sigma_q
        c_minus = (1 << i) * sigma_q
        val += c_plus * sum(comb(n - q - i - 1, w - i - 1) for w in Wq
                             if 0 <= w - i - 1 <= n - q - i - 1)
        val -= c_minus * sum(comb(n - q - i, w - i) for w in Wq
                              if 0 <= w - i <= n - q - i)
    return val


def is_sequential_valid(l, n, W_list, sigma_list):
    """
    Check that Delta_Phi(i, j) is independent of i for all j in 1..len(W_list),
    modulo 2^(l+1).
    """
    modulus = 1 << (l + 1)  # 2^(l+1) (since theta = pi/2^l, 2*pi/theta = 2^(l+1))
    for j in range(1, len(W_list) + 1):
        # Compute Delta_Phi/theta for i = 0, 1, ..., n - j
        vals = [delta_phi_over_theta(i, j, n, W_list, sigma_list) % modulus
                for i in range(0, n - j + 1)]
        if len(set(vals)) > 1:
            return False
    return True


# --- Factory parameter readout -----------------------------------------------

def factory_parameters(l, n, W_list, sigma_list):
    """
    Compute (N, k, d) for a valid sequential factory.

    N = total gate count from Eq. (eq:N_total)
    k = n - len(W_list)  (output qubits left after J steps)
    d = min weight in any W_j  (since the lowest weight gate sets distance)
    """
    J = len(W_list)
    k = n - J
    N = sum(
        sum(comb(n - j, w - 1) for w in W_list[j - 1] if w - 1 <= n - j)
        for j in range(1, J + 1)
    )
    d_min = min(min(w for w in Wj) for Wj in W_list if Wj)+1
    # Distance is the minimum gate weight (separation enforces detection)
    return N, k, d_min


# --- Search -------------------------------------------------------------------

def W_family(s, max_weight):
    """
    Arithmetic-progression family W = {w : w == 1 mod s, w <= max_weight}.
    """
    return tuple(w for w in range(1, max_weight + 1) if (w - 1) % s == 0)


def search_sequential(l, n_max=8, allow_signs=False, max_steps=None):
    """
    Search for valid sequential factories at level l, up to n_max qubits.

    Each step j has W_j from family s_j in {1, 2, 3, 4}.
    If allow_signs=True, enumerate sigma_j in {+1, -1}; else all +1.
    max_steps caps the number of sequence steps; default is l (the natural
    terminal j_f from Theorem 2).
    """
    if max_steps is None:
        max_steps = l
    s_choices = [1, 2, 3, 4]
    results = []
    for n in range(2, n_max + 1):
        for J in range(1, min(max_steps, n - 1) + 1):
            for s_tuple in product(s_choices, repeat=J):
                W_list = [W_family(s, n - j + 1) for j, s in enumerate(s_tuple, 1)]
                # Skip if any W_j is empty
                if any(not Wj for Wj in W_list):
                    continue
                sigma_options = (
                    list(product([1, -1], repeat=J)) if allow_signs
                    else [tuple([1] * J)]
                )
                for sigma_tuple in sigma_options:
                    if is_sequential_valid(l, n, W_list, list(sigma_tuple)):
                        N, k, d = factory_parameters(l, n, W_list, list(sigma_tuple))
                        if N <=k or all(set(Wj)=={1} for Wj in W_list):
                            continue
                        results.append({
                            "l": l, "n": n, "J": J, "k": k, "d": d, "N": N,
                            "s": s_tuple, "signs": sigma_tuple,
                            "W": W_list,
                        })
    return results


# --- Output -------------------------------------------------------------------

def print_factories(results, label):
    """Print a sorted list of factories."""
    print(f"\n=== {label} ===")
    if not results:
        print("  (no factories found)")
        return
    # Sort by (n, k, d, J)
    results = sorted(results, key=lambda r: (r["n"], -r["k"], -r["d"], r["J"]))
    print(f"  {'l':<3}{'n':<4}{'k':<4}{'d':<4}{'N':<5}{'J':<4}{'s':<14}{'signs':<14}")
    seen = set()
    for r in results:
        key = (r["n"], r["k"], r["d"], r["N"])
        if key in seen:
            continue
        seen.add(key)
        s_str = "(" + ",".join(str(x) for x in r["s"]) + ")"
        sign_str = "(" + ",".join("+" if x > 0 else "-" for x in r["signs"]) + ")"
        print(f"  {r['l']:<3}{r['n']:<4}{r['k']:<4}{r['d']:<4}{r['N']:<5}"
              f"{r['J']:<4}{s_str:<14}{sign_str:<14}")


# --- Main ---------------------------------------------------------------------

if __name__ == "__main__":
    L_RANGE = [2, 3, 4]
    N_MAX = 8

    print("Sequential distillation factory search")
    print(f"l in {L_RANGE}, n <= {N_MAX}, s_j in {{1,2,3,4}} per step")

    # Pass 1: positive signs only
    all_pos = []
    for l in L_RANGE:
        all_pos.extend(search_sequential(l, n_max=N_MAX, allow_signs=False))
    print_factories(all_pos, "Case A: all signs +1")

    # Pass 2: all sign patterns
    all_sign = []
    for l in L_RANGE:
        all_sign.extend(search_sequential(l, n_max=N_MAX, allow_signs=True))
    print_factories(all_sign, "Case B: signs in {+1, -1} per step")

    # Print only NEW factories from Case B (not in Case A)
    pos_keys = set((r["n"], r["k"], r["d"], r["N"]) for r in all_pos)
    new_with_signs = [r for r in all_sign
                       if (r["n"], r["k"], r["d"], r["N"]) not in pos_keys]
    print_factories(new_with_signs, "Case B - Case A: factories needing sign flips")

Sequential distillation factory search
l in [2, 3, 4], n <= 8, s_j in {1,2,3,4} per step

=== Case A: all signs +1 ===
  l  n   k   d   N    J   s             signs         
  2  4   3   2   4    1   (2)           (+)           
  2  4   2   2   6    2   (2,2)         (+,+)         
  2  4   2   2   5    2   (2,3)         (+,+)         
  2  5   4   2   8    1   (2)           (+)           
  2  5   3   2   12   2   (2,2)         (+,+)         
  3  5   2   2   14   3   (2,2,2)       (+,+,+)       
  3  5   2   2   13   3   (2,2,3)       (+,+,+)       
  2  6   5   2   16   1   (2)           (+)           
  2  6   4   2   24   2   (2,2)         (+,+)         
  3  6   3   2   28   3   (2,2,2)       (+,+,+)       
  4  6   2   2   30   4   (2,2,2,2)     (+,+,+,+)     
  4  6   2   2   29   4   (2,2,2,3)     (+,+,+,+)     
  2  7   6   2   32   1   (2)           (+)           
  2  7   6   2   16   1   (4)           (+)           
  2  7   5   2   48   2   (2,2)         (+,+)         
 

In [51]:
# --- Paper-style plotting -----------------------------------------------------
# Style matched to the asymmetric-factory figure

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.colors import BoundaryNorm, ListedColormap
import numpy as np
all_results=all_pos+all_sign
# Color and shape both encode l
COLOR_L  = {2: "#e41a1c", 3: "#377eb8", 4: "#2ca02c"}
MARKER_L = {2: "o", 3: "s", 4: "^"}
LABEL_L  = {2: r"$\ell=2$", 3: r"$\ell=3$", 4: r"$\ell=4$"}

L_VALUES = [2, 3, 4]
COLORS = [COLOR_L[l] for l in L_VALUES]
CMAP = ListedColormap(COLORS)
BOUNDS = [1.5, 2.5, 3.5, 4.5]
NORM = BoundaryNorm(BOUNDS, CMAP.N)

# Extract unique (N, k, l)
seen = set()
factories = []
for r in all_results:
    if r['N'] > 100:
        continue
    key = (r['N'], r['k'], r['l'])
    if key not in seen:
        seen.add(key)
        factories.append((r['N'], r['k'], r['l']))

print(f"Unique factories (N<=100): {len(factories)}")

# Jitter by l
L_JITTER = {2: -0.04, 3: 0.0, 4: 0.04}

fig, ax = plt.subplots(figsize=(3.4, 2.2), constrained_layout=True)

k_range = np.linspace(0.7, 7, 200)

# gamma isolines
for gamma in [1, 2, 3, 4, 5]:
    N_iso = k_range * (2 ** gamma)
    mask = N_iso <= 100
    ax.plot(N_iso[mask], k_range[mask], "--", color="#dddddd",
            linewidth=0.5, zorder=0)

# Plot
for N, k, l in factories:
    jit = L_JITTER.get(l, 0)
    ax.scatter(N * (1 + jit), k, marker=MARKER_L[l], s=14,
               c=[COLOR_L[l]], edgecolors=COLOR_L[l],
               linewidths=0.3, alpha=0.85, zorder=3)

# Colorbar
sm = plt.cm.ScalarMappable(cmap=CMAP, norm=NORM)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, boundaries=BOUNDS, ticks=L_VALUES,
                    spacing="proportional", pad=0.02,
                    fraction=0.09, aspect=24)
cbar.set_label(r"$\ell$", fontsize=7)
cbar.ax.tick_params(labelsize=6, width=0.6, length=3)
cbar.outline.set_linewidth(0.6)

# Legend
legend_elements = [
    Line2D([0],[0], marker=MARKER_L[l], color="w",
           markerfacecolor=COLOR_L[l], markeredgecolor=COLOR_L[l],
           markersize=5, label=LABEL_L[l])
    for l in L_VALUES
]
legend_elements.append(
    Line2D([0],[0], color="#bbbbbb", linestyle="--",
           linewidth=0.7, label=r"$\gamma\!=\!\log_2\frac{N}{k}$")
)
ax.legend(handles=legend_elements, loc="upper left",
          bbox_to_anchor=(0.02, 0.98),
          frameon=True, framealpha=0.92, edgecolor="#cccccc",
          fontsize=5.5, ncol=1, labelspacing=0.3,
          handletextpad=0.4, borderpad=0.4)

ax.set_xscale("log")
ax.set_xlim(3, 100)
ax.set_ylim(0.5, 7.5)
ax.set_xlabel(r"$N$", fontsize=7)
ax.set_ylabel(r"$k$", fontsize=7)
ax.set_yticks([1, 2, 3, 4, 5, 6, 7])
ax.tick_params(axis="both", labelsize=6)
ax.grid(True, linestyle=":", linewidth=0.3, alpha=0.4, zorder=0)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

fig.savefig("code_III.png", dpi=300, bbox_inches="tight")
print("saved")

Unique factories (N<=100): 37
saved


In [52]:
"""
Sequential distillation factory search (Section III).

Implements the closed-form sequential phase condition (Theorem 2):

    Delta_Phi(i, j) = [
        2^(i+1) theta sigma_j sum_{w in W_j} C(n-j-i, w-i)
      + sum_{q=1}^{j-1} 2^(i+1) theta sigma_q sum_{w in W_q} C(n-q-i-1, w-i-1)
      - sum_{q=1}^{j-1} 2^i theta sigma_q sum_{w in W_q} C(n-q-i, w-i)
    ] mod 2*pi

with theta = pi/2^l. The disentangling condition requires Delta_Phi(i, j)
to be the same for all i.
"""

from math import comb, pi
from itertools import product

import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm, ListedColormap


# --- Core verifier ------------------------------------------------------------

def delta_phi_i0(j, n, W_list, sigma_list):
    """Delta_Phi(0, j) — the only nontrivial case for j>1."""
    val = 2 * sigma_list[j-1] * sum(comb(n-j, w-1) for w in W_list[j-1] 
                                     if 0 <= w-1 <= n-j)
    for q in range(1, j):
        val += 2 * sigma_list[q-1] * sum(comb(n-q-1, w-1) for w in W_list[q-1]
                                          if 0 <= w-1 <= n-q-1)
    return val

def delta_phi_igt0_j1(i, n, W_list, sigma_list):
    """Delta_Phi(i, 1) for i >= 1 — the j=1 case."""
    val = (1 << (i+1)) * sigma_list[0] * sum(comb(n-1-i, w-i) for w in W_list[0]
                                               if 0 <= w-i <= n-1-i)
    return val

def is_sequential_valid(l, n, W_list, sigma_list):
    full_mod = 1 << (l+1)
    for j in range(1, len(W_list)+1):
        if n-j < 1:
            continue
        # Check i=0 for all j
        if delta_phi_i0(j, n, W_list, sigma_list) % full_mod != 0:
            return False
        # Check i>0 only at j=1
        if j == 1:
            for i in range(1, n):
                if delta_phi_igt0_j1(i, n, W_list, sigma_list) % full_mod != 0:
                    return False
    return True


def factory_parameters(l, n, W_list, sigma_list):
    J = len(W_list)
    k = n - J
    N = sum(
        sum(comb(n - j, w - 1) for w in W_list[j - 1] if w - 1 <= n - j)
        for j in range(1, J + 1)
    )
    d_min = min(min(w for w in Wj) for Wj in W_list if Wj) + 1
    return N, k, d_min


# --- Search -------------------------------------------------------------------

def W_family(s, max_weight):
    return tuple(w for w in range(1, max_weight + 1) if (w - 1) % s == 0)


def search_sequential(l, n_max=8, allow_signs=False, max_steps=None):
    if max_steps is None:
        max_steps = l
    s_choices = [1, 2, 3, 4]
    results = []
    for n in range(2, n_max + 1):
        for J in range(1, min(max_steps, n - 1) + 1):
            for s_tuple in product(s_choices, repeat=J):
                W_list = [W_family(s, n - j + 1) for j, s in enumerate(s_tuple, 1)]
                if any(not Wj for Wj in W_list):
                    continue
                sigma_options = (
                    list(product([1, -1], repeat=J)) if allow_signs
                    else [tuple([1] * J)]
                )
                for sigma_tuple in sigma_options:
                    if is_sequential_valid(l, n, W_list, list(sigma_tuple)):
                        N, k, d = factory_parameters(l, n, W_list, list(sigma_tuple))
                        if N <=k:
                            continue
                        results.append({
                            "l": l, "n": n, "J": J, "k": k, "d": d, "N": N,
                            "s": s_tuple, "signs": sigma_tuple,
                            "W": W_list,
                        })
    return results


def print_factories(results, label):
    print(f"\n=== {label} ===")
    if not results:
        print("  (no factories found)")
        return
    results = sorted(results, key=lambda r: (r["n"], -r["k"], -r["d"], r["J"]))
    print(f"  {'l':<3}{'n':<4}{'k':<4}{'d':<4}{'N':<5}{'J':<4}{'s':<14}{'signs':<14}")
    seen = set()
    for r in results:
        key = (r["n"], r["k"], r["d"], r["N"])
        if key in seen:
            continue
        seen.add(key)
        s_str = "(" + ",".join(str(x) for x in r["s"]) + ")"
        sign_str = "(" + ",".join("+" if x > 0 else "-" for x in r["signs"]) + ")"
        print(f"  {r['l']:<3}{r['n']:<4}{r['k']:<4}{r['d']:<4}{r['N']:<5}"
              f"{r['J']:<4}{s_str:<14}{sign_str:<14}")


# --- Paper-style plotting -----------------------------------------------------
# Style matched to the symmetric-factory figure: viridis discrete colormap,
# script-ell labels, single-column paper sizing (~3.4" wide), small fonts.

plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 8,
})


def _best_k_per_N(results):
    """
    Filter for the figure: keep d=2 factories with n ≤ 6 and k ≥ 2.
    The n cap keeps the gate count moderate (N ≤ 60) and the k filter
    drops the trivial single-qubit-output factories that aren't of
    interest as distillation factories.

    The function name is kept for compatibility with the main plot
    routine.
    """
    return [r for r in results
            if r["d"] == 2 and r["n"] <= 6 and r["k"] >= 1]


def _unique_points(results):
    """Group by (N, k, l) and collect distances seen at each (N, k, l)."""
    grouped = {}
    for r in results:
        key = (r["N"], r["k"], r["l"])
        if key not in grouped:
            grouped[key] = dict(r)
            grouped[key]["ds"] = [r["d"]]
        else:
            grouped[key]["ds"].append(r["d"])
    for r in grouped.values():
        r["ds"] = sorted(set(r["ds"]))
    return list(grouped.values())


def _discrete_l_palette(ls):
    """Discrete viridis colormap, one color per ℓ. Matches symmetric figure."""
    base = plt.get_cmap("viridis", len(ls))
    colors = [base(i) for i in range(len(ls))]
    cmap = ListedColormap(colors)
    if len(ls) == 1:
        bounds = [ls[0] - 0.5, ls[0] + 0.5]
    else:
        bounds = [ls[0] - 0.5] + [val + 0.5 for val in ls]
    norm = BoundaryNorm(bounds, cmap.N)
    return cmap, norm, bounds


def _spread_overlapping_points(results, dx=1.2):
    """Jitter points that share (N, k) horizontally so multiple ℓ values
    at the same (N, k) are all visible. Symmetric about the true N."""
    grouped = {}
    for r in results:
        key = (r["N"], r["k"])
        grouped.setdefault(key, []).append(r)

    spread = []
    for (N_val, k_val), entries in grouped.items():
        entries = sorted(entries, key=lambda r: r["l"])
        n_at = len(entries)
        # Symmetric offsets centered on 0
        if n_at == 1:
            offsets = [0.0]
        else:
            offsets = [(i - (n_at - 1) / 2) * dx for i in range(n_at)]
        for r, off in zip(entries, offsets):
            r2 = dict(r)
            r2["N_plot"] = N_val + off
            r2["k_plot"] = float(k_val)
            spread.append(r2)
    return spread


def plot_nk_distance2_by_l(results, save_path=None):
    """
    Single-column paper figure: distance-2 sequential factories in the
    (N, k) plane, filtered to keep only the BEST-k factory at each N
    (the highest k/N ratio per gate count). Concentric rings at each
    surviving (N, k) show which ℓ values achieve that best k.
    """
    # Apply the min-n-per-ℓ filter, then collapse duplicates.
    filtered = _best_k_per_N(results)
    pts = _unique_points(filtered)
    if not pts:
        print("No distance-2 results to plot")
        return None

    ls = sorted({r["l"] for r in pts})
    cmap, norm, bounds = _discrete_l_palette(ls)

    # Concentric ring sizes: smallest ℓ smallest, larger ℓ outside.
    # Sized so the outermost ring has radius < 0.5 data units in both
    # x and y (the closest (N,k) neighbors are 1 unit apart).
    base_size = 6
    size_step = 9
    sizes = {l_val: base_size + size_step * idx
             for idx, l_val in enumerate(ls)}

    fig, ax = plt.subplots(figsize=(3.4, 2.6), constrained_layout=True)

    # Group points by (N, k) and draw rings back-to-front (largest ℓ
    # first), so smaller-ℓ rings sit on top.
    grouped = {}
    for r in pts:
        grouped.setdefault((r["N"], r["k"]), []).append(r)

    for (N_val, k_val), entries in grouped.items():
        ls_present = sorted({r["l"] for r in entries})
        for l_val in sorted(ls_present, reverse=True):
            color = cmap(norm(l_val))
            ax.scatter(
                [N_val], [k_val],
                s=sizes[l_val],
                facecolor="none", edgecolor=color,
                linewidth=1.2,
                zorder=10 - l_val,
            )

    # Annotate each unique (N, k) once. Within each k-row, alternate
    # labels between above and below so close-neighbor pairs land in
    # different vertical positions and don't overlap.
    label_coords = sorted(grouped.keys(), key=lambda t: (t[1], t[0]))

    by_k = {}
    for N_val, k_val in label_coords:
        by_k.setdefault(k_val, []).append(N_val)

    for k_val, Ns_in_row in by_k.items():
        Ns_in_row = sorted(Ns_in_row)
        for idx, N_val in enumerate(Ns_in_row):
            # Even index -> below, odd index -> above. Smaller offset
            # magnitude to keep labels close to the markers.
            dy_pts = -7 if idx % 2 == 0 else 8
            ax.annotate(
                f"{N_val}",
                (N_val, k_val),
                textcoords="offset points",
                xytext=(0, dy_pts),
                fontsize=6.5, ha="center", va="center",
                zorder=5,
                bbox=dict(boxstyle="round,pad=0.08",
                          fc="white", ec="none", alpha=0.85),
            )

    ax.set_xlabel(r"$N$", fontsize=8)
    ax.set_ylabel(r"$k$", fontsize=8)
    ax.tick_params(axis="both", labelsize=7)
    ax.grid(True, linestyle="--", linewidth=0.4, alpha=0.35)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    for spine in ax.spines.values():
        spine.set_linewidth(0.6)
    ax.xaxis.set_major_locator(plt.MaxNLocator(nbins=6, integer=True))
    ax.yaxis.set_major_locator(plt.MaxNLocator(nbins=6, integer=True))
    ax.margins(x=0.06, y=0.12)

    # Discrete ℓ colorbar via ScalarMappable.
    sm = plt.cm.ScalarMappable(norm=norm, cmap=cmap)
    sm.set_array([r["l"] for r in pts])
    cbar = fig.colorbar(
        sm, ax=ax, boundaries=bounds, ticks=ls,
        spacing="proportional", pad=0.03,
    )
    cbar.set_label(r"$\ell$ (Clifford hierarchy level)", fontsize=8)
    cbar.ax.tick_params(labelsize=7, width=0.5, length=2)
    cbar.outline.set_linewidth(0.5)

    if save_path:
        fig.savefig(save_path, dpi=300, bbox_inches="tight")
        if save_path.lower().endswith(".png"):
            fig.savefig(save_path[:-4] + ".pdf", bbox_inches="tight")
        print(f"saved {save_path}")
    return fig


# --- Main ---------------------------------------------------------------------

if __name__ == "__main__":
    L_RANGE = [2, 3, 4]
    N_MAX = 8

    print("Sequential distillation factory search")
    print(f"l in {L_RANGE}, n <= {N_MAX}, s_j in {{1,2,3,4}} per step")

    all_pos = []
    for l in L_RANGE:
        all_pos.extend(search_sequential(l, n_max=N_MAX, allow_signs=False))
    print_factories(all_pos, "Case A: all signs +1")

    all_sign = []
    for l in L_RANGE:
        all_sign.extend(search_sequential(l, n_max=N_MAX, allow_signs=True))
    print_factories(all_sign, "Case B: signs in {+1, -1} per step")

    pos_keys = set((r["n"], r["k"], r["d"], r["N"]) for r in all_pos)
    new_with_signs = [r for r in all_sign
                       if (r["n"], r["k"], r["d"], r["N"]) not in pos_keys]
    print_factories(new_with_signs, "Case B - Case A: factories needing sign flips")

    #plot_nk_distance2_by_l(all_pos, save_path="code_III.png")


Sequential distillation factory search
l in [2, 3, 4], n <= 8, s_j in {1,2,3,4} per step

=== Case A: all signs +1 ===
  l  n   k   d   N    J   s             signs         
  2  3   2   2   4    1   (1)           (+)           
  2  3   1   2   6    2   (1,1)         (+,+)         
  2  4   3   2   8    1   (1)           (+)           
  2  4   3   2   4    1   (2)           (+)           
  2  4   2   2   12   2   (1,1)         (+,+)         
  2  4   2   2   6    2   (2,2)         (+,+)         
  3  4   1   2   14   3   (1,1,1)       (+,+,+)       
  2  5   4   2   16   1   (1)           (+)           
  2  5   4   2   8    1   (2)           (+)           
  2  5   3   2   24   2   (1,1)         (+,+)         
  2  5   3   2   20   2   (1,2)         (+,+)         
  2  5   3   2   16   2   (2,1)         (+,+)         
  2  5   3   2   12   2   (2,2)         (+,+)         
  3  5   2   2   28   3   (1,1,1)       (+,+,+)       
  3  5   2   2   14   3   (2,2,2)       (+,+,+)       
 

In [53]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.colors import BoundaryNorm, ListedColormap
import numpy as np
all_results=all_pos+all_sign
COLOR_L  = {2: "#e41a1c", 3: "#377eb8", 4: "#2ca02c"}
MARKER_L = {2: "o", 3: "s", 4: "^"}
LABEL_L  = {2: r"$\ell=2$", 3: r"$\ell=3$", 4: r"$\ell=4$"}
L_VALUES = [2, 3, 4]
CMAP = ListedColormap([COLOR_L[l] for l in L_VALUES])
BOUNDS = [1.5, 2.5, 3.5, 4.5]
NORM = BoundaryNorm(BOUNDS, CMAP.N)

seen = set()
factories = []
for r in all_results:
    if r['N'] > 100: continue
    key = (r['N'], r['k'], r['l'])
    if key not in seen:
        seen.add(key)
        factories.append((r['N'], r['k'], r['l']))

print(f'Unique factories (N<=100): {len(factories)}')

L_JITTER = {2: -0.04, 3: 0.0, 4: 0.04}
fig, ax = plt.subplots(figsize=(3.4, 2.2), constrained_layout=True)
k_range = np.linspace(0.7, 7, 200)

for gamma in [1,2,3,4,5]:
    N_iso = k_range * (2**gamma)
    mask = N_iso <= 100
    ax.plot(N_iso[mask], k_range[mask], "--", color="#dddddd", linewidth=0.5, zorder=0)

for N, k, l in factories:
    jit = L_JITTER.get(l, 0)
    ax.scatter(N*(1+jit), k, marker=MARKER_L[l], s=14,
               c=[COLOR_L[l]], edgecolors=COLOR_L[l],
               linewidths=0.3, alpha=0.85, zorder=3)

sm = plt.cm.ScalarMappable(cmap=CMAP, norm=NORM)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, boundaries=BOUNDS, ticks=L_VALUES,
                    spacing="proportional", pad=0.02, fraction=0.09, aspect=24)
cbar.set_label(r"$\ell$", fontsize=7)
cbar.ax.tick_params(labelsize=6, width=0.6, length=3)
cbar.outline.set_linewidth(0.6)

legend_elements = [
    Line2D([0],[0], marker=MARKER_L[l], color="w",
           markerfacecolor=COLOR_L[l], markeredgecolor=COLOR_L[l],
           markersize=5, label=LABEL_L[l])
    for l in L_VALUES
]
legend_elements.append(
    Line2D([0],[0], color="#bbbbbb", linestyle="--",
           linewidth=0.7, label=r"$\gamma\!=\!\log_2\frac{N}{k}$")
)
ax.legend(handles=legend_elements, loc="upper left",
          bbox_to_anchor=(0.02, 0.98),
          frameon=True, framealpha=0.92, edgecolor="#cccccc",
          fontsize=5.5, ncol=1, labelspacing=0.3,
          handletextpad=0.4, borderpad=0.4)

LANDMARKS = [
    (8,  3, r"$[[8,3,2]]$",   (-10,   5)),
    (8,  4, r"$[[8,4,2]]$",   (4,  10)),
    (12, 2, r"$[[12,2,2]]$",  (-10,  5)),
    (12, 3, r"$[[12,3,2]]$",  (4,   10)),
    (14, 1, r"$[[14,1,2]]$",  (4,  -5)),
    (14, 2, r"$[[14,2,2]]$",  (4,   -10)),
    (15, 1, r"$[[15,1,3]]$",  (-25, 10)),
]

for N, k, label, xytext in LANDMARKS:
    ax.annotate(
        label, (N, k),
        xytext=xytext, textcoords="offset points",
        fontsize=5, color="#222222", zorder=6,
        bbox=dict(boxstyle="round,pad=0.12", fc="white",
                  ec="#aaa", lw=0.3, alpha=0.9),
        arrowprops=dict(arrowstyle="-", color="#aaa",
                        lw=0.4, shrinkA=0, shrinkB=1),
    )
ax.set_xlim(3, 100)
ax.set_ylim(0.5, 7.5)
ax.set_xlabel(r"$N$", fontsize=7)
ax.set_ylabel(r"$k$", fontsize=7)
ax.set_yticks([1,2,3,4,5,6,7])
ax.tick_params(axis="both", labelsize=6)
ax.grid(True, linestyle=":", linewidth=0.3, alpha=0.4, zorder=0)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

fig.savefig("code_III.png", dpi=300, bbox_inches="tight")
print("saved")

Unique factories (N<=100): 96
saved


In [60]:
# Label the gamma isolines at their upper ends using a darker color for contrast.
for gamma in [1, 2, 3, 4, 5]:
    N_iso = k_range * (2 ** gamma)
    mask = N_iso <= 100
    if not np.any(mask):
        continue
    ax.annotate(
        rf"$\gamma={gamma}$",
        (N_iso[mask][-1], k_range[mask][-1]),
        xytext=(3, 1),
        textcoords="offset points",
        color="#666666",
        fontsize=5,
        ha="left",
        va="bottom",
        zorder=5,
    )

fig.savefig("code_III.png", dpi=300, bbox_inches="tight")
print("saved")

saved


In [61]:
ax.legend(handles=legend_elements, loc="lower right",
          bbox_to_anchor=(0.98, 0.02),
          frameon=True, framealpha=0.92, edgecolor="#cccccc",
          fontsize=5.5, ncol=1, labelspacing=0.3,
          handletextpad=0.4, borderpad=0.4)

fig.savefig("code_III.png", dpi=300, bbox_inches="tight")
print("saved")

saved
